# 서울 도심까지 거리 변수 추가
- 카카오 로컬 API를 이용해 시군구 + 번지 기반 주소를 좌표로 변환
- Haversine 공식으로 광화문(서울 도심 기준)까지의 거리(km) 계산
- `서울도심거리` 컬럼 추가

In [ ]:
import pandas as pd
import requests
import time
import math

In [ ]:
# 카카오 API 등록
KAKAO_API_KEY = "18a57db71d648659b950bd26a83fbcb3"

# 광화문 좌표 (서울 도심 기준)
GWANGHWAMUN_LAT = 37.5759
GWANGHWAMUN_LON = 126.9769

In [ ]:
# 변수 추가할 CSV 데이터 불러오기
df = pd.read_csv("new_city.csv", encoding="utf-8-sig")
print("데이터 크기:", df.shape)
df.head()

In [ ]:
#주소 조합
# 예: "경기도 성남시 분당구 백현동 525"
df["full_address"] = df["시군구"].astype(str).str.strip() + " " + df["번지"].astype(str).str.strip()

print("주소 예시:")
print(df["full_address"].head())

In [ ]:
# 카카오 API → (위도, 경도) 변환 함수
def get_coordinates(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}

    # 1차: 정확한 주소 검사
    params = {"query": address, "analyze_type": "exact"}
    try:
        res = requests.get(url, headers=headers, params=params, timeout=5)
        docs = res.json().get("documents", [])
        if docs:
            return float(docs[0]["y"]), float(docs[0]["x"])

        # 2차: 유사한 주소 검사
        params["analyze_type"] = "similar"
        res = requests.get(url, headers=headers, params=params, timeout=5)
        docs = res.json().get("documents", [])
        if docs:
            return float(docs[0]["y"]), float(docs[0]["x"])

    except Exception as e:
        print(f"  [오류] {address}: {e}")

    return None, None

In [ ]:
# Haversine 거리 계산 함수
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371
    d_lat = math.radians(lat2 - lat1)
    d_lon = math.radians(lon2 - lon1)
    a = (math.sin(d_lat / 2) ** 2
         + math.cos(math.radians(lat1))
         * math.cos(math.radians(lat2))
         * math.sin(d_lon / 2) ** 2)
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

In [ ]:
# 고유 주소 리스트화 (중복 최소화)
unique_addresses = df["full_address"].unique()
print(f"\n고유 주소 수: {len(unique_addresses)}개")

address_to_coords = {}

# 고유 주소마다 카카오 API 호출해서 위도/경도 받아서 딕셔너리에 저장
for i, addr in enumerate(unique_addresses):
    lat, lon = get_coordinates(addr)
    address_to_coords[addr] = (lat, lon)

    if (i + 1) % 50 == 0:
        print(f"  {i + 1}/{len(unique_addresses)} 완료")

    time.sleep(0.1)  # API 호출 제한 준수 (초당 10회)

print("  좌표 변환 완료")

In [ ]:
# 위도·경도 컬럼 추가
df["위도"] = df["full_address"].apply(lambda a: address_to_coords[a][0])
df["경도"] = df["full_address"].apply(lambda a: address_to_coords[a][1])

print(f"좌표 변환 실패 건수: {df['위도'].isna().sum()}건")

In [ ]:
# 광화문까지 거리 계산 + 변수값 수집
def calc_distance(row):
    if pd.isna(row["위도"]) or pd.isna(row["경도"]):
        return None
    return haversine_distance(row["위도"], row["경도"],
                              GWANGHWAMUN_LAT, GWANGHWAMUN_LON)

df["서울도심거리"] = df.apply(calc_distance, axis=1)

print("\n[거리 변수 통계]")
print(df["서울도심거리"].describe())

In [ ]:
# CSV 저장
df_result = df.drop(columns=["full_address", "위도", "경도"])
df_result.to_csv("gyeyang_merged.csv", index=False, encoding="utf-8-sig")

print("\n저장 완료")
print("추가된 컬럼: 서울도심거리")
df_result[["시군구", "번지", "서울도심거리"]].head(10)